## 1_3_1 Saccade Classification (rebuild)

Minimal bootstrap notebook:
- define `data_path`
- load metadata generated by `1_2_SLEAP_processing.ipynb`
- load the selected eye parquet based on `eye_with_least_low_confidence`
- optional debug plot of `Ellipse.Center.X` in browser


In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go


# symbols to use ✅ ℹ️ ⚠️ ❗

In [9]:
# Set up variables and load data
##########################################################################

data_path = Path(
    "/Users/rancze/Documents/Data/vestVR/20250409_Cohort3_rotation/Visual_mismatch_day4/B6J2783-2025-04-28T14-57-30"
)

debug = True

save_path = data_path.parent / f"{data_path.name}_processedData"
downsampled_output_dir = save_path / "downsampled_data"
metadata_path = downsampled_output_dir / "saccade_input_metadata.json"

if not metadata_path.exists():
    raise FileNotFoundError(
        f"Metadata not found at {metadata_path}. Run 1_2_SLEAP_processing.ipynb first."
    )

with open(metadata_path, "r") as f:
    metadata = json.load(f)

eye_with_least_low_confidence = metadata.get("eye_with_least_low_confidence")
if eye_with_least_low_confidence not in {"VideoData1", "VideoData2"}:
    raise ValueError(
        "metadata['eye_with_least_low_confidence'] is missing or invalid. "
        "Expected 'VideoData1' or 'VideoData2'."
    )

video_outputs = metadata.get("outputs", {}).get(eye_with_least_low_confidence, {})
video_parquet_path = video_outputs.get("resampled_parquet")

if video_parquet_path is None:
    raise ValueError(
        f"No resampled_parquet path found in metadata for {eye_with_least_low_confidence}."
    )

video_parquet_path = Path(video_parquet_path)
if not video_parquet_path.exists():
    raise FileNotFoundError(
        f"Parquet file not found at {video_parquet_path} for {eye_with_least_low_confidence}."
    )

VideoData = pd.read_parquet(video_parquet_path).reset_index(drop=True)

if debug: 
    print(f"Loaded metadata: {metadata_path}")
    print(f"Selected eye/video: {eye_with_least_low_confidence}")
    print(f"Loaded parquet: {video_parquet_path}")
    print(f"VideoData shape: {VideoData.shape}")
    display(VideoData.head())
else:
    print(f"✅ Loaded selected eye/video: {eye_with_least_low_confidence}")


Loaded metadata: /Users/rancze/Documents/Data/vestVR/20250409_Cohort3_rotation/Visual_mismatch_day4/B6J2783-2025-04-28T14-57-30_processedData/downsampled_data/saccade_input_metadata.json
Selected eye/video: VideoData1
Loaded parquet: /Users/rancze/Documents/Data/vestVR/20250409_Cohort3_rotation/Visual_mismatch_day4/B6J2783-2025-04-28T14-57-30_processedData/downsampled_data/VideoData1_resampled.parquet
VideoData shape: (1954474, 5)


,Seconds,Ellipse.Center.X,Ellipse.Center.Y,Pupil.Diameter,blink_flag
0,347566.426784,3.362375,-5.301148,35.234209,0
1,347566.427784,3.402714,-5.289737,35.185107,0
2,347566.428784,3.443063,-5.278323,35.135993,0
3,347566.429784,3.483402,-5.266912,35.086891,0
4,347566.430784,3.523742,-5.255501,35.037789,0


In [5]:
# Debug plot: Ellipse.Center.X in browser
if debug:
    required_cols = {"Seconds", "Ellipse.Center.X"}
    missing_cols = [c for c in required_cols if c not in VideoData.columns]
    if missing_cols:
        raise KeyError(
            f"VideoData is missing required columns for debug plot: {missing_cols}"
        )

    time_rel_s = VideoData["Seconds"] - VideoData["Seconds"].iloc[0]

    fig = go.Figure(
        data=[
            go.Scatter(
                x=time_rel_s,
                y=VideoData["Ellipse.Center.X"],
                mode="lines",
                name="Ellipse.Center.X",
                line=dict(width=1),
            )
        ]
    )
    fig.update_layout(
        title=f"Ellipse.Center.X ({eye_with_least_low_confidence})",
        xaxis_title="Relative time (s)",
        yaxis_title="Ellipse.Center.X (px)",
        template="plotly_white",
    )
    fig.show(renderer="browser")
